# Causal Elasticity with EconML LinearDML
To run segmented own-price elasticity with a dedicated causal control profile that is intentionally leaner than the forecasting feature profile.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from retail_forecasting.causal_dml import (
    ElasticityRunConfig,
    fit_elasticity_pipeline,
    load_elasticity_estimates,
    load_elasticity_run_summary,
)

In [ ]:
config = ElasticityRunConfig(
    segment_level="product",
    feature_profile="lean",
    min_samples=250,
    min_non_null_pairs=250,
    min_unique_price_values=8,
    min_log_price_std=0.01,
    min_log_units_std=0.01,
    epsilon=1e-3,
    use_llm_features=False,
    nuisance_model="random-forest",
)

artifacts = fit_elasticity_pipeline(config)
artifacts

In [ ]:
summary = load_elasticity_run_summary()
estimates = load_elasticity_estimates()

print("Segmentation level:", summary.get("segmentation_level"))
print("Feature profile:", summary.get("feature_profile_used"))
print("Segments attempted:", summary.get("total_segments_attempted"))
print("Successful fits:", summary.get("successful_fits"))
print("Skipped fits:", summary.get("skipped_fits"))
print("LLM augmentation used:", summary.get("llm_feature_augmentation_used"))
print("Min unique price threshold:", summary.get("min_unique_price_values_threshold"))
print("Min log(price) std threshold:", summary.get("min_log_price_std_threshold"))
print("Inference warning count:", summary.get("inference_warning_count"))

estimates.head()

In [ ]:
successful = estimates.loc[estimates["fit_status"] == "success"].copy()
skipped = estimates.loc[estimates["fit_status"] != "success"].copy()

print("Successful segment count:", len(successful))
print("Skipped segment count:", len(skipped))

if not successful.empty:
    display(successful.nsmallest(10, columns=["elasticity_estimate"]))
    display(successful.nlargest(10, columns=["elasticity_estimate"]))

if not skipped.empty:
    display(skipped[["segment_key", "fit_status", "skip_reason"]].head(20))

In [ ]:
output_paths = summary.get("output_paths", {})
hist_path = output_paths.get("elasticity_histogram_png")
ci_path = output_paths.get("elasticity_ci_plot_png")
top_csv_path = output_paths.get("elasticity_top_segments_csv")

print("Histogram path:", hist_path)
print("CI plot path:", ci_path)
print("Top segments CSV path:", top_csv_path)

if hist_path:
    display(Image(filename=hist_path))
if ci_path:
    display(Image(filename=ci_path))
if top_csv_path:
    display(pd.read_csv(top_csv_path).head(20))